# Stage 3 - clean-data baselines (Random Forest and 1D-CNN)

Train both models on the prepared arrays and evaluate on the held-out test set. Read macro-F1, the per-class report and the confusion matrix - not overall accuracy. The CNN uses balanced class weights only where the dataset config asks for it.

In [ ]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
pd.set_option('display.width', 140)

# The two datasets are treated identically: every step below runs the SAME
# code for both. The only dataset-specific code in the project is each
# dataset's loader (adversec/datasets/ciciov.py, road.py).
DATASETS = ['ciciov2024', 'road']

## Setup - device and a small loader

In [ ]:
import torch, joblib
from sklearn.utils.class_weight import compute_class_weight
from adversec.models import build_random_forest, CNN1D, train_cnn
from adversec.evaluation import evaluate_model
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)

def load_arrays(name):
    a = np.load(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz')
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    return a['X_train'], a['y_train'], a['X_test'], a['y_test'], classes

## Random Forest baseline (both datasets)

In [ ]:
rf_models = {}
for name in DATASETS:
    Xtr, ytr, Xte, yte, classes = load_arrays(name)
    rf = build_random_forest(); rf.fit(Xtr, ytr); rf_models[name] = rf
    evaluate_model(yte, rf.predict(Xte), classes, model_name=f'{name} - Random Forest')

## 1D-CNN baseline (both datasets)
Watch the per-epoch loss, then read the same metric set as the RF.

In [ ]:
cnn_models = {}
for name in DATASETS:
    Xtr, ytr, Xte, yte, classes = load_arrays(name)
    cfg = config.load_dataset_config(name)
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    print(f'\n=== training CNN: {name} (class weights: {bool(cw is not None)}) ===')
    cnn = CNN1D(n_features=Xtr.shape[1], n_classes=len(classes))
    cnn = train_cnn(cnn, Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw)
    cnn_models[name] = cnn
    cnn.eval()
    with torch.no_grad():
        pred = cnn(torch.tensor(Xte, dtype=torch.float32, device=DEVICE)).argmax(1).cpu().numpy()
    evaluate_model(yte, pred, classes, model_name=f'{name} - 1D-CNN')